# ETL — V1DD release 1196 (single-notebook)

Writes the full V1DD 1196 release into the common-connectivity schemas under one notebook, project `v1dd`.

**Two DataSets inside project `v1dd`:**
- `v1dd_1196_em` — every EM soma in `soma_and_cell_type_1196.feather` (DataItem id = soma `id`).
- `v1dd_1196_func` — every functional ROI in `snr_by_cell.feather` (DataItem id = `f"{volume}-{column}-{plane}-{roi}"`).

**Additional cohort DataSets (subsets of `v1dd_1196_em`):**
- `v1dd_1196_proofread_axons` — `proofread_axon_list_1196.npy`.
- `v1dd_1196_proofread_dendrites` — `proofread_dendrite_list_1196.npy`.

**Additional cohort DataSet (subset of `v1dd_1196_func`):**
- `v1dd_1196_func_coregistered` — functional ROIs that appear in `coregistration_1196.feather`.

Sections marked **TODO** are skeletons only; we will fill them together. Each section ends with an **open questions** list.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa

from connects_common_connectivity.models import (
    AlgorithmRun,
    CellCellConnectivityLong,
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    CellToCellMapping,
    Cluster,
    ClusterHierarchy,
    ClusterMembership,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    MappingSet,
    Modality,
    SpatialLocation,
    Unit,
)
from connects_common_connectivity.io import write_models
from connects_common_connectivity.io.arrow_utils import build_cell_feature_matrix_schema
from connects_common_connectivity.io.write_utils import walk_ancestors
from deltalake import write_deltalake

In [2]:
DATA_ROOT   = Path("/data/v1dd_1196")
OUTPUT_ROOT = "../scratch/v1dd_1196_v1/"
PROJECT_ID  = "v1dd"
RELEASE     = "1196"

DATASET_EM   = "v1dd_1196_em"
DATASET_FUNC = "v1dd_1196_func"
DATASET_PROOFREAD_AXON = "v1dd_1196_proofread_axons"
DATASET_PROOFREAD_DEND = "v1dd_1196_proofread_dendrites"
DATASET_FUNC_COREG     = "v1dd_1196_func_coregistered"

HIERARCHY_ID_V1DD   = "v1dd_cell_types"
HIERARCHY_ID_MINNIE = "minnie65_csm_cell_types"   # for comparison only

FS_EM_SOMA_GEOM  = "v1dd_em_soma_geometry"
FS_FUNC_QC       = "v1dd_func_qc"
FS_FUNC_POSITION = "v1dd_func_imaging_position"

In [3]:
# Sanity check: every expected input file is on disk.
expected = [
    "data_description.json",
    "subject.json",
    "soma_and_cell_type_1196.feather",
    "proofread_axon_list_1196.npy",
    "proofread_dendrite_list_1196.npy",
    "snr_by_cell.feather",
    "coregistration_1196.feather",
    "syn_df_all_to_proofread_to_all_1196.feather",
    "syn_label_df_all_to_proofread_to_all_1196.feather",
    "cell_cell_correlations_by_stimulus.feather",
    "cell_cell_correlations_by_stimulus_coregistered.feather",
]
missing = [f for f in expected if not (DATA_ROOT / f).exists()]
assert not missing, f"Missing input files: {missing}"
print("All input files present.")

All input files present.


## Master decision — `DataItem.id = pt_root_id`

Every downstream file in this release is keyed by `pt_root_id`; only `soma_and_cell_type_1196.feather` carries the per-detection soma `id`. We therefore use `str(pt_root_id)` as the EM `DataItem.id` for the whole notebook and treat soma-centroid data as features attached to the cell.

### Where each id appears

| File | soma `id` | `pt_root_id` |
|---|:---:|:---:|
| `soma_and_cell_type_1196.feather` | ✅ | ✅ |
| `proofread_axon_list_1196.npy` | — | ✅ |
| `proofread_dendrite_list_1196.npy` | — | ✅ |
| `coregistration_1196.feather` | — | ✅ |
| `syn_df_all_to_proofread_to_all_1196.feather` | — | ✅ (`pre_pt_root_id`, `post_pt_root_id`) |
| `cell_cell_correlations_by_stimulus_coregistered.feather` | — | ✅ |
| `snr_by_cell.feather`, `cell_cell_correlations_by_stimulus.feather` | — | — (functional ROI tuples) |

### Key counts from `soma_and_cell_type` (207,455 rows)

| quantity | value |
|---|---:|
| unique soma `id` | 207,455 |
| unique `pt_root_id` | 163,064 |
| rows with `pt_root_id == 0` (orphan detections) | 3,835 |
| rows with non-zero `pt_root_id` | 203,620 |
| unique non-zero `pt_root_id` | 163,063 |
| `pt_root_id`s with > 1 soma row | 19,615 (~12 %) |
| max soma rows for a single `pt_root_id` | 184 |

### Policy

- **EM `DataItem.id = str(pt_root_id)`** (one row per segment), `name = str(pt_root_id)`.
- **Drop `pt_root_id == 0` rows** — they cannot be referenced from any other file and so cannot be cohort-associated or linked.
- **Collapse multi-soma `pt_root_id`s** to one DataItem by picking the soma row with the largest `volume` (largest nucleus detection is the most plausible primary soma); the other rows are dropped from the cell-features matrix. Number of collapsed cells: 19,615; rows discarded: 207,455 − 163,063 − 3,835 = 40,557.
- **Downstream joins are direct lookups** on `pt_root_id` everywhere. No `pt_root_id → soma_id` resolution step is needed in §2, §4, §4, §4, §4.

This matches the way the rest of the V1DD release is keyed and aligns with Minnie's nucleus-per-segment convention (the 12 % multi-detection rate is the only thing that differs — Minnie's `nucleus_detection_lookup_v1` already does the collapse at the source).

## 1. `DataSet` rows

Five DataSet rows under `project_id="v1dd"`. Provenance comes from `data_description.json`; `publication` points at the V1DD physiology repository.

| id | modality | parent |
|---|---|---|
| `v1dd_1196_em` | `ELECTRON_MICROSCOPY` | — |
| `v1dd_1196_proofread_axons` | `ELECTRON_MICROSCOPY` | subset of `v1dd_1196_em` |
| `v1dd_1196_proofread_dendrites` | `ELECTRON_MICROSCOPY` | subset of `v1dd_1196_em` |
| `v1dd_1196_func` | `CALCIUM_IMAGING` | — |
| `v1dd_1196_func_coregistered` | `CALCIUM_IMAGING` | subset of `v1dd_1196_func` |

In [4]:
V1DD_PUBLICATION = "https://github.com/AllenInstitute/v1dd_physiology"

datasets = [
    DataSet(
        id=DATASET_EM,
        name="V1DD release 1196 — EM somas",
        publication=V1DD_PUBLICATION,
        modality=Modality.ELECTRON_MICROSCOPY.value,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_PROOFREAD_AXON,
        name="V1DD release 1196 — proofread axons cohort",
        publication=V1DD_PUBLICATION,
        modality=Modality.ELECTRON_MICROSCOPY.value,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_PROOFREAD_DEND,
        name="V1DD release 1196 — proofread dendrites cohort",
        publication=V1DD_PUBLICATION,
        modality=Modality.ELECTRON_MICROSCOPY.value,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_FUNC,
        name="V1DD release 1196 — functional 2P ROIs",
        publication=V1DD_PUBLICATION,
        modality=Modality.CALCIUM_IMAGING.value,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_FUNC_COREG,
        name="V1DD release 1196 — coregistered functional ROIs cohort",
        publication=V1DD_PUBLICATION,
        modality=Modality.CALCIUM_IMAGING.value,
        project_id=PROJECT_ID,
    ),
]

result = write_models(datasets, output_root=OUTPUT_ROOT)
print(f"DataSet rows written: {result.rows_written}")

DataSet rows written: 5


In [5]:
ds_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataset/")
      .filter(pl.col("project_id") == PROJECT_ID)
)
print("shape:", ds_verify.shape)
print(ds_verify.select(["id", "name", "modality", "publication"]).to_pandas().to_string(index=False))

expected_ids = {DATASET_EM, DATASET_PROOFREAD_AXON, DATASET_PROOFREAD_DEND,
                DATASET_FUNC, DATASET_FUNC_COREG}
got_ids = set(ds_verify["id"].to_list())
assert expected_ids <= got_ids, f"missing DataSet ids: {expected_ids - got_ids}"
assert ds_verify["id"].n_unique() == ds_verify.shape[0], "duplicate DataSet ids"
modalities = dict(zip(ds_verify["id"].to_list(), ds_verify["modality"].to_list()))
for em_id in (DATASET_EM, DATASET_PROOFREAD_AXON, DATASET_PROOFREAD_DEND):
    assert modalities[em_id] == Modality.ELECTRON_MICROSCOPY.value, em_id
for fn_id in (DATASET_FUNC, DATASET_FUNC_COREG):
    assert modalities[fn_id] == Modality.CALCIUM_IMAGING.value, fn_id
print("\nOK — 5 DataSet rows for project v1dd.")

shape: (5, 5)
                           id                                                    name            modality                                       publication
  v1dd_1196_func_coregistered V1DD release 1196 — coregistered functional ROIs cohort     CALCIUM_IMAGING https://github.com/AllenInstitute/v1dd_physiology
               v1dd_1196_func                  V1DD release 1196 — functional 2P ROIs     CALCIUM_IMAGING https://github.com/AllenInstitute/v1dd_physiology
v1dd_1196_proofread_dendrites          V1DD release 1196 — proofread dendrites cohort ELECTRON_MICROSCOPY https://github.com/AllenInstitute/v1dd_physiology
    v1dd_1196_proofread_axons              V1DD release 1196 — proofread axons cohort ELECTRON_MICROSCOPY https://github.com/AllenInstitute/v1dd_physiology
                 v1dd_1196_em                            V1DD release 1196 — EM somas ELECTRON_MICROSCOPY https://github.com/AllenInstitute/v1dd_physiology

OK — 5 DataSet rows for project v1dd.


## 2. EM `DataItem`s and `DataItemDataSetAssociation`s

Per the master decision: one EM `DataItem` per unique non-zero `pt_root_id` in `soma_and_cell_type_1196.feather`. `id = name = str(pt_root_id)`. Where multiple soma rows share a `pt_root_id`, we keep the soma row with the largest `volume` (the largest nucleus detection per segment).

Associations written:
- Every kept EM cell → `v1dd_1196_em`.
- Cells whose `pt_root_id` ∈ `proofread_axon_list_1196.npy` → also `v1dd_1196_proofread_axons`. 1164/1210 axon roots are in the soma catalog; the remaining 46 are logged and skipped (proofread axon fragments without a soma centroid in this release).
- Cells whose `pt_root_id` ∈ `proofread_dendrite_list_1196.npy` → also `v1dd_1196_proofread_dendrites` (63986/63986 present).

`DataItem.neuroglancer_link` is intentionally left null: V1DD does not ship a per-cell neuroglancer URL template in this release. Once the V1DD team publishes one it can be backfilled in a focused follow-up rather than guessed at here.


In [6]:
soma_df  = pd.read_feather(DATA_ROOT / "soma_and_cell_type_1196.feather")
axon_ids = np.load(DATA_ROOT / "proofread_axon_list_1196.npy", allow_pickle=True)
dend_ids = np.load(DATA_ROOT / "proofread_dendrite_list_1196.npy", allow_pickle=True)

print("soma_df shape       :", soma_df.shape)
print("pt_root_id unique   :", soma_df['pt_root_id'].nunique())
print("id unique           :", soma_df['id'].nunique())
print("axon ids unique     :", len(set(axon_ids.tolist())))
print("dend ids unique     :", len(set(dend_ids.tolist())))
print("axon ∩ soma roots   :", len(set(axon_ids.tolist()) & set(soma_df['pt_root_id'].tolist())))
print("dend ∩ soma roots   :", len(set(dend_ids.tolist()) & set(soma_df['pt_root_id'].tolist())))

soma_df shape       : (207455, 11)
pt_root_id unique   : 163064
id unique           : 207455
axon ids unique     : 1210
dend ids unique     : 63986
axon ∩ soma roots   : 1164
dend ∩ soma roots   : 63986


In [7]:
# --- Collapse soma catalog to one row per non-zero pt_root_id (largest-volume soma) ---
soma_keep = (
    soma_df
    .query("pt_root_id != 0")
    .sort_values("volume", ascending=False)
    .drop_duplicates(subset="pt_root_id", keep="first")
    .reset_index(drop=True)
)
print(f"soma rows kept (one per pt_root_id): {len(soma_keep):,}")
assert soma_keep["pt_root_id"].is_unique
assert (soma_keep["pt_root_id"] != 0).all()

em_ids = [str(int(rid)) for rid in soma_keep["pt_root_id"]]
em_id_set = set(em_ids)

# --- Cohort membership: keep only proofread roots that exist as DataItems ---
axon_root_set = {int(x) for x in axon_ids.tolist()}
dend_root_set = {int(x) for x in dend_ids.tolist()}

axon_ids_in_soma = sorted(r for r in axon_root_set if str(r) in em_id_set)
axon_ids_missing = sorted(axon_root_set - set(axon_ids_in_soma))
dend_ids_in_soma = sorted(r for r in dend_root_set if str(r) in em_id_set)
dend_ids_missing = sorted(dend_root_set - set(dend_ids_in_soma))

print(
    f"proofread axon roots: {len(axon_root_set):,} input -> "
    f"{len(axon_ids_in_soma):,} associated; {len(axon_ids_missing):,} skipped (no soma)"
)
if axon_ids_missing:
    preview = axon_ids_missing[:5]
    print(f"  skipped axon root ids (first 5 of {len(axon_ids_missing)}): {preview}")
print(
    f"proofread dend roots: {len(dend_root_set):,} input -> "
    f"{len(dend_ids_in_soma):,} associated; {len(dend_ids_missing):,} skipped (no soma)"
)

# --- DataItem rows (id = name = str(pt_root_id)) ---
em_dataitems = [
    DataItem(id=cid, name=cid, project_id=PROJECT_ID) for cid in em_ids
]
n_di = write_models(em_dataitems, output_root=OUTPUT_ROOT).rows_written
print(f"DataItem rows appended: {n_di:,} (sent {len(em_dataitems):,})")

# --- Base association: every EM cell -> v1dd_1196_em ---
em_assoc = [
    DataItemDataSetAssociation(
        dataitem_id=cid, dataset_id=DATASET_EM, project_id=PROJECT_ID,
    )
    for cid in em_ids
]
n_em = write_models(em_assoc, output_root=OUTPUT_ROOT).rows_written
print(f"Associations -> {DATASET_EM}: {n_em:,}")

# --- Cohort association: proofread axons ---
axon_assoc = [
    DataItemDataSetAssociation(
        dataitem_id=str(rid), dataset_id=DATASET_PROOFREAD_AXON, project_id=PROJECT_ID,
    )
    for rid in axon_ids_in_soma
]
n_axon = write_models(axon_assoc, output_root=OUTPUT_ROOT).rows_written
print(f"Associations -> {DATASET_PROOFREAD_AXON}: {n_axon:,}")

# --- Cohort association: proofread dendrites ---
dend_assoc = [
    DataItemDataSetAssociation(
        dataitem_id=str(rid), dataset_id=DATASET_PROOFREAD_DEND, project_id=PROJECT_ID,
    )
    for rid in dend_ids_in_soma
]
n_dend = write_models(dend_assoc, output_root=OUTPUT_ROOT).rows_written
print(f"Associations -> {DATASET_PROOFREAD_DEND}: {n_dend:,}")


soma rows kept (one per pt_root_id): 163,063
proofread axon roots: 1,210 input -> 1,164 associated; 46 skipped (no soma)
  skipped axon root ids (first 5 of 46): [864691132549572162, 864691132572564252, 864691132593374461, 864691132596563511, 864691132599901759]
proofread dend roots: 63,986 input -> 63,986 associated; 0 skipped (no soma)


DataItem rows appended: 0 (sent 163,063)


Associations -> v1dd_1196_em: 163,063
Associations -> v1dd_1196_proofread_axons: 1,164


Associations -> v1dd_1196_proofread_dendrites: 63,986


In [8]:
# --- Verification: EM DataItem and cohort association slices ---
di_v = (
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
      .filter(pl.col("project_id") == PROJECT_ID)
)
print("dataitem/ (project=v1dd) shape:", di_v.shape)
print(di_v.head(3).to_pandas().to_string(index=False))
assert di_v["id"].n_unique() == di_v.shape[0], "duplicate DataItem ids"
assert di_v.shape[0] == len(em_ids), (
    f"expected {len(em_ids):,} DataItems, got {di_v.shape[0]:,}"
)
assert set(di_v["id"].to_list()) == em_id_set, "DataItem ids mismatch source set"

assoc = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
      .filter(pl.col("project_id") == PROJECT_ID)
)
by_ds = (
    assoc.group_by("dataset_id").len()
         .sort("dataset_id")
         .to_pandas()
)
print("\nassociations per dataset_id (project=v1dd):")
print(by_ds.to_string(index=False))

counts = dict(zip(by_ds["dataset_id"], by_ds["len"]))
assert counts.get(DATASET_EM) == len(em_ids), (
    f"em assoc count mismatch: got {counts.get(DATASET_EM)} expected {len(em_ids)}"
)
assert counts.get(DATASET_PROOFREAD_AXON) == len(axon_ids_in_soma), (
    f"axon assoc count mismatch: got {counts.get(DATASET_PROOFREAD_AXON)} "
    f"expected {len(axon_ids_in_soma)}"
)
assert counts.get(DATASET_PROOFREAD_DEND) == len(dend_ids_in_soma), (
    f"dend assoc count mismatch: got {counts.get(DATASET_PROOFREAD_DEND)} "
    f"expected {len(dend_ids_in_soma)}"
)

# Cohort dataitem_ids must be a subset of v1dd_1196_em
em_assoc_ids = set(
    assoc.filter(pl.col("dataset_id") == DATASET_EM)["dataitem_id"].to_list()
)
for cohort in (DATASET_PROOFREAD_AXON, DATASET_PROOFREAD_DEND):
    cohort_ids = set(
        assoc.filter(pl.col("dataset_id") == cohort)["dataitem_id"].to_list()
    )
    assert cohort_ids <= em_assoc_ids, (
        f"{cohort} has {len(cohort_ids - em_assoc_ids)} ids not in {DATASET_EM}"
    )

print("\nOK — EM DataItems + 3 cohort association slices verified.")


dataitem/ (project=v1dd) shape: (163063, 4)
                id               name neuroglancer_link project_id
864691132787860302 864691132787860302              None       v1dd
864691132845344210 864691132845344210              None       v1dd
864691132756766666 864691132756766666              None       v1dd

associations per dataset_id (project=v1dd):
                   dataset_id    len
                 v1dd_1196_em 163063
    v1dd_1196_proofread_axons   1164
v1dd_1196_proofread_dendrites  63986

OK — EM DataItems + 3 cohort association slices verified.


## 3. EM soma `CellFeatureMatrix` (`v1dd_em_soma_geometry`)

Numeric features per EM `DataItem`, sourced from `soma_and_cell_type_1196.feather` after the §2 collapse (one row per non-zero `pt_root_id`, largest-volume soma):

| feature | dtype | unit | description |
|---|---|---|---|
| `pt_position_x`        | `<i8` | `voxel`          | raw EM-acquisition voxel index, x (column) |
| `pt_position_y`        | `<i8` | `voxel`          | raw EM-acquisition voxel index, y (row) |
| `pt_position_z`        | `<i8` | `voxel`          | raw EM-acquisition voxel index, z (section) |
| `pt_position_trform_x` | `<f8` | `MICRONS_LENGTH` | V1DD-aligned soma centroid, x (µm) |
| `pt_position_trform_y` | `<f8` | `MICRONS_LENGTH` | V1DD-aligned soma centroid, y — cortical depth, pia ≈ 0 (µm) |
| `pt_position_trform_z` | `<f8` | `MICRONS_LENGTH` | V1DD-aligned soma centroid, z (µm) |
| `volume`               | `<f8` | `MICRONS_CUBED`  | soma volume from the EM segmentation (µm³) |

Writes: seven `CellFeatureDefinition` rows, one `CellFeatureSet` (`v1dd_em_soma_geometry`), one `CellFeatureMatrix` pointer row, and the wide parquet at `cellfeatures/v1dd_em_soma_geometry/` (partitioned by `(project_id, feature_set_id)`).

`pt_position_trform_*` values are stored as features (alongside the raw voxel `pt_position_*` and `volume`) rather than as a separate `SingleCellReconstruction` / `SpatialLocation` record — schema-level reference-space metadata is not yet pinned down for V1DD, and the feature-matrix route mirrors how `etl_minnie_02` handles its standard-transform coordinates. Values are converted from the source's native nm to µm (÷ 1000) so the unit matches the schema enum `MICRONS_LENGTH`. The frame is V1DD-local (pia ≈ y=0); it is **not** Allen CCF.

Coordinate-system assumption (carried forward to §4 / §4): the unprefixed `pt_position_*` columns in `soma_and_cell_type_1196`, `coregistration_1196`, and the synapse tables share the same EM voxel index space.


In [9]:
# --- CellFeatureDefinition rows (7 features: 3 voxel ints + 3 µm trform floats + 1 µm³ float) ---
em_soma_geom_fds = [
    CellFeatureDefinition(
        id="pt_position_x",
        description=(
            "Raw EM-acquisition voxel index along the x axis (column) of the V1DD 1196 EM "
            "volume. Same voxel coordinate space as the unprefixed pt_position_* columns "
            "in coregistration_1196 and the synapse tables. Registered (µm-scale) "
            "coordinates are carried by the pt_position_trform_* features in this set."
        ),
        unit="voxel",
        data_type="<i8",
        project_id=PROJECT_ID,
        feature_set_id=FS_EM_SOMA_GEOM,
    ),
    CellFeatureDefinition(
        id="pt_position_y",
        description=(
            "Raw EM-acquisition voxel index along the y axis (row) of the V1DD 1196 EM volume. "
            "See pt_position_x for the coordinate-frame caveat."
        ),
        unit="voxel",
        data_type="<i8",
        project_id=PROJECT_ID,
        feature_set_id=FS_EM_SOMA_GEOM,
    ),
    CellFeatureDefinition(
        id="pt_position_z",
        description=(
            "Raw EM-acquisition voxel index along the z axis (section) of the V1DD 1196 EM volume. "
            "See pt_position_x for the coordinate-frame caveat."
        ),
        unit="voxel",
        data_type="<i8",
        project_id=PROJECT_ID,
        feature_set_id=FS_EM_SOMA_GEOM,
    ),
    CellFeatureDefinition(
        id="pt_position_trform_x",
        description=(
            "V1DD-aligned soma centroid, x axis, in micrometres. Source: "
            "pt_position_trform_x in soma_and_cell_type_1196.feather (native nm), "
            "divided by 1000. Reference frame is V1DD-local (pia ≈ y=0); not Allen CCF. "
            "Convention mirrors the standard_transform feature set in etl_minnie_02."
        ),
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f8",
        project_id=PROJECT_ID,
        feature_set_id=FS_EM_SOMA_GEOM,
    ),
    CellFeatureDefinition(
        id="pt_position_trform_y",
        description=(
            "V1DD-aligned soma centroid, y axis, in micrometres. Cortical depth axis: "
            "y ≈ 0 at pia, larger values deeper into cortex (white-matter direction). "
            "Source nm ÷ 1000."
        ),
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f8",
        project_id=PROJECT_ID,
        feature_set_id=FS_EM_SOMA_GEOM,
    ),
    CellFeatureDefinition(
        id="pt_position_trform_z",
        description=(
            "V1DD-aligned soma centroid, z axis, in micrometres. Source nm ÷ 1000. "
            "See pt_position_trform_x for the reference-frame note."
        ),
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f8",
        project_id=PROJECT_ID,
        feature_set_id=FS_EM_SOMA_GEOM,
    ),
    CellFeatureDefinition(
        id="volume",
        description=(
            "Soma volume from the V1DD 1196 EM segmentation, in cubic micrometres. "
            "Values are fractional (float64), consistent with a physical-units conversion "
            "from raw voxel counts rather than the voxel count itself."
        ),
        unit=Unit.MICRONS_CUBED.value,
        data_type="<f8",
        project_id=PROJECT_ID,
        feature_set_id=FS_EM_SOMA_GEOM,
    ),
]
n_fd = write_models(em_soma_geom_fds, output_root=OUTPUT_ROOT).rows_written
print(f"CellFeatureDefinition rows written: {n_fd}")

# --- CellFeatureSet ---
cfs_em_soma_geom = CellFeatureSet(
    id=FS_EM_SOMA_GEOM,
    description=(
        "EM soma geometry per cell for V1DD release 1196: raw EM-acquisition voxel index "
        "(pt_position_x/y/z), V1DD-aligned soma centroid in µm (pt_position_trform_x/y/z; "
        "pia ≈ y=0, not Allen CCF), and soma volume in µm³. One row per pt_root_id; where "
        "multiple soma detections share a root, the largest-volume soma is kept (see §2)."
    ),
    feature_definition_ids=[fd.id for fd in em_soma_geom_fds],
    extraction_method=(
        "Loaded soma_and_cell_type_1196.feather; dropped pt_root_id == 0; kept the "
        "largest-volume soma per pt_root_id; copied pt_position_{x,y,z} and volume verbatim; "
        "divided pt_position_trform_{x,y,z} by 1000 to convert from nm to µm."
    ),
    project_id=PROJECT_ID,
)
n_fs = write_models([cfs_em_soma_geom], output_root=OUTPUT_ROOT).rows_written
print(f"CellFeatureSet rows written: {n_fs}")

# --- Wide-form parquet (rows = EM DataItems, cols in def order + scope cols) ---
em_soma_wide = pd.DataFrame({
    "id":                   soma_keep["pt_root_id"].astype("int64").astype(str).values,
    "pt_position_x":        soma_keep["pt_position_x"].astype("int64").values,
    "pt_position_y":        soma_keep["pt_position_y"].astype("int64").values,
    "pt_position_z":        soma_keep["pt_position_z"].astype("int64").values,
    "pt_position_trform_x": (soma_keep["pt_position_trform_x"].astype("float64") / 1000.0).values,
    "pt_position_trform_y": (soma_keep["pt_position_trform_y"].astype("float64") / 1000.0).values,
    "pt_position_trform_z": (soma_keep["pt_position_trform_z"].astype("float64") / 1000.0).values,
    "volume":               soma_keep["volume"].astype("float64").values,
})
em_soma_wide["project_id"]     = PROJECT_ID
em_soma_wide["feature_set_id"] = FS_EM_SOMA_GEOM

schema_wide = build_cell_feature_matrix_schema(
    cfs_em_soma_geom, em_soma_geom_fds, cell_index_column="id",
)
table_wide = pa.Table.from_pandas(em_soma_wide, schema=schema_wide, preserve_index=False)

write_deltalake(
    OUTPUT_ROOT + f"cellfeatures/{FS_EM_SOMA_GEOM}/", table_wide,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
print(f"Wide cellfeatures/{FS_EM_SOMA_GEOM}/ written: {table_wide.shape}")

# --- CellFeatureMatrix pointer row ---
output_abs = Path(OUTPUT_ROOT).resolve()
cfm_em_soma_geom = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FS_EM_SOMA_GEOM}",
    feature_set_id=FS_EM_SOMA_GEOM,
    parquet_path=f"file://{output_abs}/cellfeatures/{FS_EM_SOMA_GEOM}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
n_cfm = write_models([cfm_em_soma_geom], output_root=OUTPUT_ROOT).rows_written
print(f"CellFeatureMatrix rows written: {n_cfm}")


CellFeatureDefinition rows written: 7
CellFeatureSet rows written: 1


Wide cellfeatures/v1dd_em_soma_geometry/ written: (163063, 10)
CellFeatureMatrix rows written: 1


In [10]:
# --- Verification: defs, set, wide parquet, matrix pointer ---
expected_voxel = {"pt_position_x", "pt_position_y", "pt_position_z"}
expected_trform = {"pt_position_trform_x", "pt_position_trform_y", "pt_position_trform_z"}
expected_fds = expected_voxel | expected_trform | {"volume"}

cfd_v = (
    pl.read_delta(OUTPUT_ROOT + "cellfeaturedefinition/")
      .filter((pl.col("project_id") == PROJECT_ID)
              & (pl.col("feature_set_id") == FS_EM_SOMA_GEOM))
)
print("cellfeaturedefinition shape:", cfd_v.shape)
print(cfd_v.select(["id", "unit", "data_type"]).to_pandas().to_string(index=False))
assert cfd_v.shape[0] == 7, cfd_v.shape
assert set(cfd_v["id"].to_list()) == expected_fds
units = dict(zip(cfd_v["id"].to_list(), cfd_v["unit"].to_list()))
dtypes = dict(zip(cfd_v["id"].to_list(), cfd_v["data_type"].to_list()))
for c in expected_voxel:
    assert units[c] == "voxel", (c, units[c])
    assert dtypes[c] == "<i8", (c, dtypes[c])
for c in expected_trform:
    assert units[c] == Unit.MICRONS_LENGTH.value, (c, units[c])
    assert dtypes[c] == "<f8", (c, dtypes[c])
assert units["volume"] == Unit.MICRONS_CUBED.value
assert dtypes["volume"] == "<f8"

cfs_v = (
    pl.read_delta(OUTPUT_ROOT + "cellfeatureset/")
      .filter(pl.col("id") == FS_EM_SOMA_GEOM)
)
print("\ncellfeatureset shape:", cfs_v.shape)
assert cfs_v.shape[0] == 1
assert set(cfs_v["feature_definition_ids"][0]) == expected_fds

wide_v = (
    pl.read_delta(OUTPUT_ROOT + f"cellfeatures/{FS_EM_SOMA_GEOM}/")
      .filter(pl.col("project_id") == PROJECT_ID)
)
print("\ncellfeatures/v1dd_em_soma_geometry/ shape:", wide_v.shape)
print(wide_v.head(3).to_pandas().to_string(index=False))
assert wide_v.shape[0] == len(soma_keep), (wide_v.shape, len(soma_keep))
assert wide_v["id"].n_unique() == wide_v.shape[0], "duplicate ids in wide parquet"
assert set(wide_v["id"].to_list()) == em_id_set, "wide parquet ids != EM DataItem ids"
for col in expected_fds | {"id", "project_id", "feature_set_id"}:
    assert col in wide_v.columns, f"missing column: {col}"

# Sanity: trform µm values are in the right physical range (cortical scale, not nm).
for c in expected_trform:
    s = wide_v[c]
    lo, hi = s.min(), s.max()
    assert -2000.0 <= lo and hi <= 2000.0, (
        f"{c} out of expected µm range; got [{lo}, {hi}]. Did the ÷1000 conversion happen?"
    )
print(f"\ntrform µm ranges: "
      f"x=[{wide_v['pt_position_trform_x'].min():.1f},{wide_v['pt_position_trform_x'].max():.1f}], "
      f"y=[{wide_v['pt_position_trform_y'].min():.1f},{wide_v['pt_position_trform_y'].max():.1f}], "
      f"z=[{wide_v['pt_position_trform_z'].min():.1f},{wide_v['pt_position_trform_z'].max():.1f}]")

cfm_v = (
    pl.read_delta(OUTPUT_ROOT + "cellfeaturematrix/")
      .filter(pl.col("feature_set_id") == FS_EM_SOMA_GEOM)
)
print("\ncellfeaturematrix shape:", cfm_v.shape)
print(cfm_v.to_pandas().to_string(index=False))
assert cfm_v.shape[0] == 1
assert cfm_v["parquet_path"][0].startswith("file://")
assert cfm_v["parquet_path"][0].endswith(f"/cellfeatures/{FS_EM_SOMA_GEOM}/")
assert cfm_v["cell_index_column"][0] == "id"

print("\nOK — v1dd_em_soma_geometry defs (7), set, wide parquet, and matrix pointer verified.")


cellfeaturedefinition shape: (7, 8)
                  id           unit data_type
       pt_position_x          voxel       <i8
       pt_position_y          voxel       <i8
       pt_position_z          voxel       <i8
pt_position_trform_x MICRONS_LENGTH       <f8
pt_position_trform_y MICRONS_LENGTH       <f8
pt_position_trform_z MICRONS_LENGTH       <f8
              volume  MICRONS_CUBED       <f8

cellfeatureset shape: (1, 5)



cellfeatures/v1dd_em_soma_geometry/ shape: (163063, 10)
                id  pt_position_x  pt_position_y  pt_position_z  pt_position_trform_x  pt_position_trform_y  pt_position_trform_z      volume project_id        feature_set_id
864691132787860302         469713         389862         704565           -406.052640            180.651165            460.354708 1939.461544       v1dd v1dd_em_soma_geometry
864691132845344210         452718         329102         704430           -421.315252            120.594411            463.051808 1431.969259       v1dd v1dd_em_soma_geometry
864691132756766666         328946         525352         112320           -545.325862            152.696986           -152.567598 1408.035021       v1dd v1dd_em_soma_geometry

trform µm ranges: x=[-921.0,902.4], y=[-62.1,884.8], z=[-465.1,562.5]

cellfeaturematrix shape: (1, 5)
                        id        feature_set_id                                                     parquet_path cell_index_column project

## 4. V1DD cell-type taxonomy — `ClusterHierarchy` + `ClusterMembership`

Two-level taxonomy from `soma_and_cell_type` columns; ids are kept verbatim from source (per the "never reformat ids" rule):
- Level 0 (root): `neuron`.
- Level 1: `cell_type_coarse` ∈ {`E`, `I`}.
- Level 2 (excitatory leaves, 8): `L2-IT`, `L3-IT`, `L4-IT`, `L5-IT`, `L5-ET`, `L5-NP`, `L6-IT`, `L6-CT`.
- Level 2 (inhibitory leaves, 4): `DTC`, `ITC`, `PTC`, `STC`.

Writes (all scoped via `write_models`):
- `cluster/` — 15 rows under `hierarchy_id = "v1dd_cell_types"`.
- `algorithmrun/` — 1 row, `id = "v1dd_1196_cell_type_classifier"`. The V1DD release does not document a specific classification method (`data_description.json` lists project/investigators only), so the row is a minimal placeholder pointing at `input_dataset = v1dd_1196_em`.
- `clusterhierarchy/` — 1 row, `id = "v1dd_cell_types"`, `root = "neuron"`, `run = "v1dd_1196_cell_type_classifier"`.
- `clustermembership/` — one row per (cell × ancestor) for every labelled EM `DataItem`, denormalized via `walk_ancestors` so consumers can filter at any level without a recursive join. `(project_id, hierarchy_id)`-scoped per `write_spec.py`.

### Source-data findings (probed)

- 49,192 of the 163,063 collapsed EM cells carry a label (`cell_type` non-null); 113,871 are unlabelled.
- Coarse and fine labels are perfectly co-present: **0 cells have coarse-only labels.** The "parent-only membership" policy (for cells with `coarse` but no leaf) is documented for completeness but emits no rows in this release.
- All 19,615 multi-soma `pt_root_id`s have **0 disagreements** on `cell_type` / `cell_type_coarse`, so the §2 largest-volume collapse is label-safe regardless of tiebreak.
- Expected membership row count: 49,192 cells × 3 ancestor levels = **147,576 rows**.

### V1DD vs MICrONS Minnie taxonomy comparison

Verified against `data/microns1412/cluster` (Minnie CSM taxonomy in `etl_minnie_03_*`).

| Aspect | Minnie65 (`minnie65_csm_cell_types`) | V1DD (`v1dd_cell_types`) |
|---|---|---|
| Root | `neuron` | `neuron` |
| Level-1 ids | `glutamatergic`, `gabaergic` | `E`, `I` |
| Excitatory leaves | `L2IT`, `L3IT`, `L4IT`, `L5IT`, `L5ET`, `L5NP`, `L6IT`, `L6CT`, `L6SP` (9) | `L2-IT`, `L3-IT`, `L4-IT`, `L5-IT`, `L5-ET`, `L5-NP`, `L6-IT`, `L6-CT` (8) |
| Inhibitory leaves | `PTC`, `DTC`, `ITC`, `STC` (4) | `PTC`, `DTC`, `ITC`, `STC` (4) |
| Naming | no hyphens (`L4IT`) | hyphenated (`L4-IT`) |
| Excitatory-only differences | has `L6SP` (subplate) | no `L6SP` |

Inhibitory leaves are *identical strings* across the two taxonomies; excitatory leaves are *semantically aligned* but differ in spelling (hyphen). Because `Cluster.id` is scoped per `hierarchy_id`, the spellings can diverge safely.

### Intentional omissions

- **`HierarchyCategory` not written.** `HierarchyCategory` is a global, shared vocabulary (`class`, `subclass`, `cluster`, …) and a scoped overwrite from one taxonomy notebook would clobber rows owned by other taxonomies. Same precedent as `etl_minnie_03`; needs a global-dedup append helper before any taxonomy notebook should write there. `Cluster.hierarchy_category` is left null.
- **`Cluster.hex_color` left null** — no palette is shipped with the V1DD source, and the schema's hex-color validator demands `^#?[0-9A-Fa-f]{6}$`, so inventing colors here would be guesswork.
- **`ClusterMembership.membership_score`, `.probability`, `.distance` left null** — the V1DD labels are hard assignments with no soft-membership metric.

### Future work (deferred)

A `MappingSet` + `ClusterToClusterMapping` linking `v1dd_cell_types` ↔ `minnie65_csm_cell_types` would be a useful follow-up artifact: inhibitory pairs are trivial (id match across hierarchies), and excitatory needs a manual map (`L4-IT` ↔ `L4IT`, etc., with `L6SP` unmapped). Not in scope for this notebook.


In [11]:
# Quick label distribution sanity
print("cell_type_coarse counts:")
print(soma_df['cell_type_coarse'].value_counts(dropna=False))
print()
print("cell_type counts:")
print(soma_df['cell_type'].value_counts(dropna=False))
print()
print("coarse vs fine (cross-tab):")
print(pd.crosstab(soma_df['cell_type_coarse'].fillna('NA'),
                  soma_df['cell_type'].fillna('NA')))

cell_type_coarse counts:
cell_type_coarse
None    158263
E        42495
I         6697
Name: count, dtype: int64

cell_type counts:
cell_type
None     158263
L6-CT     11260
L4-IT      7955
L3-IT      6361
L6-IT      6044
L5-IT      5090
L2-IT      3073
PTC        2951
L5-ET      2013
DTC        1933
ITC        1090
STC         723
L5-NP       699
Name: count, dtype: int64

coarse vs fine (cross-tab):


cell_type          DTC   ITC  L2-IT  L3-IT  L4-IT  L5-ET  L5-IT  L5-NP  L6-CT  \
cell_type_coarse                                                                
E                    0     0   3073   6361   7955   2013   5090    699  11260   
I                 1933  1090      0      0      0      0      0      0      0   
NA                   0     0      0      0      0      0      0      0      0   

cell_type         L6-IT      NA   PTC  STC  
cell_type_coarse                            
E                  6044       0     0    0  
I                     0       0  2951  723  
NA                    0  158263     0    0  


In [12]:
# --- Taxonomy definition (verbatim from soma_and_cell_type) ---
ROOT_ID_V1DD = "neuron"
RUN_ID_V1DD = "v1dd_1196_cell_type_classifier"

E_LEAVES = ["L2-IT", "L3-IT", "L4-IT", "L5-IT", "L5-ET", "L5-NP", "L6-IT", "L6-CT"]
I_LEAVES = ["DTC", "ITC", "PTC", "STC"]

# Build (parent, level) once; derive children from parents.
parent_of: dict[str, str | None] = {ROOT_ID_V1DD: None, "E": ROOT_ID_V1DD, "I": ROOT_ID_V1DD}
parent_of.update({leaf: "E" for leaf in E_LEAVES})
parent_of.update({leaf: "I" for leaf in I_LEAVES})

level_of = {ROOT_ID_V1DD: 0, "E": 1, "I": 1}
level_of.update({leaf: 2 for leaf in E_LEAVES + I_LEAVES})

children_of: dict[str, list[str]] = {}
for cid, parent in parent_of.items():
    if parent is not None:
        children_of.setdefault(parent, []).append(cid)

cluster_rows = [
    Cluster(
        id=cid,
        hierarchy_id=HIERARCHY_ID_V1DD,
        parent=parent_of[cid],
        children=(children_of.get(cid) or None),
        level=level_of[cid],
    )
    for cid in parent_of
]
assert len(cluster_rows) == 15, f"expected 15 cluster rows, got {len(cluster_rows)}"
n_clu = write_models(cluster_rows, output_root=OUTPUT_ROOT).rows_written
print(f"Cluster rows written ({HIERARCHY_ID_V1DD}): {n_clu}")

# --- AlgorithmRun (minimal placeholder; classifier method not documented in release) ---
run_row = AlgorithmRun(
    id=RUN_ID_V1DD,
    algorithm_name=(
        "V1DD release 1196 cell-type labels "
        "(classification method not documented in release)"
    ),
    algorithm_version=RELEASE,
    input_dataset=DATASET_EM,
)
n_run = write_models([run_row], output_root=OUTPUT_ROOT).rows_written
print(f"AlgorithmRun rows written: {n_run}")

# --- ClusterHierarchy ---
hierarchy_row = ClusterHierarchy(
    id=HIERARCHY_ID_V1DD,
    run=RUN_ID_V1DD,
    root=ROOT_ID_V1DD,
    clusters=[c.id for c in cluster_rows],
)
n_h = write_models([hierarchy_row], output_root=OUTPUT_ROOT).rows_written
print(f"ClusterHierarchy rows written: {n_h}")

# --- ClusterMembership: one row per (labelled cell × ancestor), via walk_ancestors ---
labelled = soma_keep.dropna(subset=["cell_type"])
print(f"labelled cells: {len(labelled):,} (unlabelled skipped: {len(soma_keep) - len(labelled):,})")
assert set(labelled["cell_type"].unique()) <= set(E_LEAVES + I_LEAVES), (
    "unexpected cell_type values: "
    f"{sorted(set(labelled['cell_type'].unique()) - set(E_LEAVES + I_LEAVES))}"
)

memberships: list[ClusterMembership] = []
for cid_int, leaf in zip(labelled["pt_root_id"].to_numpy(), labelled["cell_type"].to_numpy()):
    cell_id = str(int(cid_int))
    leaf_id = str(leaf)
    for ancestor_id, _is_leaf in walk_ancestors(leaf_id, parent_of):
        memberships.append(ClusterMembership(
            item=cell_id,
            cluster=ancestor_id,
            hierarchy_id=HIERARCHY_ID_V1DD,
            project_id=PROJECT_ID,
        ))
print(f"ClusterMembership rows built: {len(memberships):,}")
assert len(memberships) == len(labelled) * 3, "expected 3 rows per labelled cell"

n_mem = write_models(memberships, output_root=OUTPUT_ROOT).rows_written
print(f"ClusterMembership rows written: {n_mem:,}")


Cluster rows written (v1dd_cell_types): 15


AlgorithmRun rows written: 1


ClusterHierarchy rows written: 1
labelled cells: 49,192 (unlabelled skipped: 113,871)


ClusterMembership rows built: 147,576


ClusterMembership rows written: 147,576


In [13]:
# --- Verification: cluster/, algorithmrun/, clusterhierarchy/, clustermembership/ ---
clu_v = (
    pl.read_delta(OUTPUT_ROOT + "cluster/")
      .filter(pl.col("hierarchy_id") == HIERARCHY_ID_V1DD)
)
print(f"cluster/ ({HIERARCHY_ID_V1DD}) shape:", clu_v.shape)
print(clu_v.select(["id", "parent", "level"]).sort(["level", "id"]).to_pandas().to_string(index=False))
assert clu_v.shape[0] == 15, clu_v.shape
got_ids = set(clu_v["id"].to_list())
assert got_ids == set(parent_of), f"missing: {set(parent_of) - got_ids}; extra: {got_ids - set(parent_of)}"

clu_parent = dict(zip(clu_v["id"].to_list(), clu_v["parent"].to_list()))
for cid, expected_parent in parent_of.items():
    got_parent = clu_parent[cid]
    assert (got_parent is None and expected_parent is None) or got_parent == expected_parent, (
        f"parent mismatch for {cid}: got {got_parent!r}, expected {expected_parent!r}"
    )
clu_level = dict(zip(clu_v["id"].to_list(), clu_v["level"].to_list()))
for cid, expected_level in level_of.items():
    assert clu_level[cid] == expected_level, (cid, clu_level[cid], expected_level)

run_v = pl.read_delta(OUTPUT_ROOT + "algorithmrun/").filter(pl.col("id") == RUN_ID_V1DD)
print(f"\nalgorithmrun/ ({RUN_ID_V1DD}) shape:", run_v.shape)
assert run_v.shape[0] == 1
assert run_v["input_dataset"][0] == DATASET_EM
assert run_v["algorithm_version"][0] == RELEASE

h_v = pl.read_delta(OUTPUT_ROOT + "clusterhierarchy/").filter(pl.col("id") == HIERARCHY_ID_V1DD)
print(f"\nclusterhierarchy/ ({HIERARCHY_ID_V1DD}) shape:", h_v.shape)
assert h_v.shape[0] == 1
assert h_v["root"][0] == ROOT_ID_V1DD
assert h_v["run"][0] == RUN_ID_V1DD
assert set(h_v["clusters"][0]) == set(parent_of)

mem_v = (
    pl.read_delta(OUTPUT_ROOT + "clustermembership/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("hierarchy_id") == HIERARCHY_ID_V1DD))
)
print(f"\nclustermembership/ ({PROJECT_ID}, {HIERARCHY_ID_V1DD}) shape:", mem_v.shape)
n_labelled = labelled.shape[0]
assert mem_v.shape[0] == n_labelled * 3, (mem_v.shape, n_labelled * 3)

# Every item must be a registered EM DataItem, and every cluster must be a known cluster id
assert set(mem_v["item"].to_list()) <= em_id_set, "membership references unknown DataItem ids"
assert set(mem_v["cluster"].to_list()) <= set(parent_of), "membership references unknown cluster ids"

# Every labelled cell must have exactly 3 ancestor rows (leaf + coarse + root) and no duplicates.
by_item = mem_v.group_by("item").len()
assert (by_item["len"] == 3).all(), "some cells do not have exactly 3 membership rows"
pair_n = mem_v.group_by(["item", "cluster"]).len()
assert (pair_n["len"] == 1).all(), "duplicate (item, cluster) membership rows detected"

# Row counts per cluster id match expectation: each leaf appears once per labelled cell of that
# label; each coarse-level cluster appears once per labelled cell of that coarse; root appears
# once per labelled cell total.
per_cluster_df = mem_v.group_by("cluster").len()
per_cluster = dict(zip(
    per_cluster_df["cluster"].to_list(),
    per_cluster_df["len"].to_list(),
))
leaf_counts = labelled["cell_type"].value_counts().to_dict()
for leaf, expected in leaf_counts.items():
    assert per_cluster[str(leaf)] == expected, (leaf, per_cluster[str(leaf)], expected)
coarse_counts = labelled["cell_type_coarse"].value_counts().to_dict()
for coarse, expected in coarse_counts.items():
    assert per_cluster[str(coarse)] == expected, (coarse, per_cluster[str(coarse)], expected)
assert per_cluster[ROOT_ID_V1DD] == n_labelled, (per_cluster[ROOT_ID_V1DD], n_labelled)

print(f"\nmembership rows per cluster (sorted):")
for cid in [ROOT_ID_V1DD, "E", "I"] + E_LEAVES + I_LEAVES:
    print(f"  {cid:<8} {per_cluster.get(cid, 0):>6}")

print("\nOK — V1DD taxonomy: 15 clusters + 1 AlgorithmRun + 1 ClusterHierarchy + "
      f"{n_labelled * 3:,} ClusterMembership rows verified.")


cluster/ (v1dd_cell_types) shape: (15, 9)
    id parent  level
neuron   None      0
     E neuron      1
     I neuron      1
   DTC      I      2
   ITC      I      2
 L2-IT      E      2
 L3-IT      E      2
 L4-IT      E      2
 L5-ET      E      2
 L5-IT      E      2
 L5-NP      E      2
 L6-CT      E      2
 L6-IT      E      2
   PTC      I      2
   STC      I      2

algorithmrun/ (v1dd_1196_cell_type_classifier) shape: (1, 9)

clusterhierarchy/ (v1dd_cell_types) shape: (1, 4)

clustermembership/ (v1dd, v1dd_cell_types) shape: (147576, 7)

membership rows per cluster (sorted):
  neuron    49192
  E         42495
  I          6697
  L2-IT      3073
  L3-IT      6361
  L4-IT      7955
  L5-IT      5090
  L5-ET      2013
  L5-NP       699
  L6-IT      6044
  L6-CT     11260
  DTC        1933
  ITC        1090
  PTC        2951
  STC         723

OK — V1DD taxonomy: 15 clusters + 1 AlgorithmRun + 1 ClusterHierarchy + 147,576 ClusterMembership rows verified.


## 5. Functional `DataItem`s + cohorts

Source: `snr_by_cell.feather` (4458 ROIs). One DataItem per ROI, `id = f"{volume}-{column}-{plane}-{roi}"`, all associated with `v1dd_1196_func`. Functional cells that appear in `coregistration_1196.feather` get an additional association to `v1dd_1196_func_coregistered`.

**Open questions:**
1. Id scheme — `f"{volume}-{column}-{plane}-{roi}"` produces e.g. `"3-1-0-143"`. Alternative: `f"v1dd_func_{volume}_{column}_{plane}_{roi}"` to avoid id collisions with EM `id`s (which are also short ints, but live in the same `DataItem` table partitioned by project_id only). EM ids are integers like `228132`; collision risk is real if a functional id like `3-1-0-143` is parsed by a downstream consumer as a string vs int. Decision: keep all DataItem ids as strings (per the no-cast rule) and use the hyphen form; mixed-shape ids are fine.
2. `DataItem.name` for functional cells — repeat the id, or something more human-readable (`f"v{volume} c{column} p{plane} roi{roi}"`)?
3. `snr_by_cell` covers 4458 ROIs but `coregistration_1196.feather` lists only 571 EM↔func pairs. Should we register all 4458 as DataItems, or only the union of `snr_by_cell` and `coregistration` (which equals `snr_by_cell` if every coreg ROI has an SNR row — needs verification)?
4. Are there ROIs that appear in `coregistration` but **not** in `snr_by_cell`? If yes, they need DataItem rows too — `snr_by_cell` would not be the right registration source.

In [14]:
snr_df   = pd.read_feather(DATA_ROOT / "snr_by_cell.feather")
coreg_df = pd.read_feather(DATA_ROOT / "coregistration_1196.feather")

snr_key   = snr_df[['volume','column','plane','roi']].drop_duplicates()
coreg_key = coreg_df[['volume','column','plane','roi']].drop_duplicates()

print("snr_df rows                :", len(snr_df))
print("snr_df unique func cells   :", len(snr_key))
print("coreg_df rows              :", len(coreg_df))
print("coreg unique func cells    :", len(coreg_key))
merged = snr_key.merge(coreg_key, how='outer', indicator=True)
print("in snr only                :", (merged['_merge']=='left_only').sum())
print("in coreg only              :", (merged['_merge']=='right_only').sum())
print("in both                    :", (merged['_merge']=='both').sum())

snr_df rows                : 4458
snr_df unique func cells   : 4458
coreg_df rows              : 571
coreg unique func cells    : 565
in snr only                : 3899
in coreg only              : 6
in both                    : 559


In [15]:
# TODO: build functional DataItem rows (key = volume-column-plane-roi).
# TODO: build associations to DATASET_FUNC for all; to DATASET_FUNC_COREG for coregistered subset.
# TODO: write + verify.
pass

## 6. Functional feature sets

Two `CellFeatureSet`s on the functional DataItems:

- **`v1dd_func_qc`** — single scalar `snr` (dtype `<f8`).
- **`v1dd_func_imaging_position`** — three int features `volume`, `column`, `plane`, `roi` (dtype `<i8`). These uniquely identify the ROI in the imaging acquisition; storing them as features (in addition to encoding them in the DataItem id) makes filtering and grouping queryable without parsing the id string.

**Open questions:**
1. Is `volume / column / plane / roi` really a *feature* in the schema sense, or metadata that belongs elsewhere? The schema has no slot for "imaging acquisition coordinates" on `DataItem`. Putting them in a feature set is a workable but slightly awkward fit. Alternative: skip `v1dd_func_imaging_position` entirely and let downstream consumers parse the id string.
2. Units of `snr` — dimensionless ratio; set `units = "ratio"` or leave null?
3. Is there a functional analogue of `SingleCellReconstruction` / `SpatialLocation` (i.e. is there an estimated CCF coordinate for each ROI)? If yes, that would belong on the EM-soma feature matrix in §3 (or a parallel functional one), not here.

In [16]:
# TODO: build CellFeatureDefinitions for snr; for volume/column/plane/roi.
# TODO: build CellFeatureSets and CellFeatureMatrix rows.
# TODO: write wide parquets at cellfeatures/v1dd_func_qc/ and cellfeatures/v1dd_func_imaging_position/.
pass

## 7. Coregistration — `CellToCellMapping` (EM ↔ functional)

Source: `coregistration_1196.feather`. One `MappingSet(id="v1dd_1196_coregistration")` with `source_dataset=v1dd_1196_em`, `target_dataset=v1dd_1196_func`. One `CellToCellMapping` row per (`pt_root_id`, ROI) pair; `score`/`probability` left null (no confidence scores in this table).

Mapping is **not** unique on either side — see exploration cell below.

In [17]:
print("coreg_df rows                  :", len(coreg_df))
print("unique pt_root_id              :", coreg_df['pt_root_id'].nunique())
print("unique (vol,col,pln,roi) tuples:", coreg_df.drop_duplicates(['volume','column','plane','roi']).shape[0])
print()
print("top pt_root_ids by # ROI matches:")
print(coreg_df.groupby('pt_root_id').size().sort_values(ascending=False).head(5))
print()
print("top ROIs by # pt_root_id matches:")
print(coreg_df.groupby(['volume','column','plane','roi']).size().sort_values(ascending=False).head(5))

coreg_df rows                  : 571
unique pt_root_id              : 553
unique (vol,col,pln,roi) tuples: 565

top pt_root_ids by # ROI matches:
pt_root_id
864691132770893729    3
864691132609937787    3
864691132565276366    3
864691132709485099    2
864691132773390355    2
dtype: int64

top ROIs by # pt_root_id matches:
volume  column  plane  roi
3       1       2      127    2
                3      98     2
                4      90     2
                3      89     2
                1      267    2
dtype: int64


**Open questions:**
1. `CellToCellMapping.id` — auto-generate as `f"{source_cell}__{target_cell}__{plane}"` (the `plane` disambiguates the repeated EM↔func pairs)? Or hash? Schema requires `id`.
2. The mapping has no confidence info, but `coregistration_1196.feather` includes ROIs at multiple optical planes for the same `pt_root_id`. Each such pair becomes a separate `CellToCellMapping` row (current plan).
3. The repository pattern in §4b for `MappingSet` / `CellToCellMapping` — predicates and partitioning. Need to confirm against the prompt guide / any existing example notebook.

(The `pt_root_id → soma_id` question is gone: EM DataItems are keyed by `pt_root_id` directly per the master decision.)

In [18]:
# TODO: build MappingSet row.
# TODO: build CellToCellMapping rows after joining pt_root_id -> soma DataItem id.
# TODO: write mappingset/ and celltocellmapping/ with correct predicates.
pass

## 8. Synapses — `CellCellConnectivityLong`

Source: `syn_df_all_to_proofread_to_all_1196.feather` (8.2M rows) + `syn_label_df_all_to_proofread_to_all_1196.feather` (6.7M tag rows, indexed by synapse `id`).

Aggregate per (`pre_pt_root_id`, `post_pt_root_id`) pair into:
- `synapse_count` — number of synapses (count, dimensionless).
- `synapse_size_sum` — total `size` (voxel-count; units to confirm).
- *(optional)* `spine_synapse_count` — count of synapses tagged `spine`.

Write to its own subdirectory per §4g: `cellcellconnectivitylong_proofread_to_proofread/` (folder name from the source feather). Pre/post cell ids are EM `DataItem` ids — i.e. `str(pt_root_id)` directly, no join needed.

**Open question — defer decision:** *Should raw per-synapse rows get their own schema?* Today `CellCellConnectivityLong` collapses to one row per cell pair, which loses per-synapse position, size, and label information. Two paths:
- Keep aggregated only; ship raw rows as a parquet sidecar outside the common schema.
- Propose a new `Synapse` class (slots: id, pre_cell, post_cell, ctr_position, size, tag) — would require a schema PR.

Leaving this **open**; the skeleton implements the aggregated form only.

**Other open questions:**
1. What `measurement_type` enum value covers `synapse_count` and `synapse_size_sum`? Need to read `SynapticMeasurementType` enum values.
2. Unit for `synapse_size_sum` — `size` is in voxels (need confirmation); convert to nm³ or leave as voxel counts?
3. Most synapse endpoints (≈4.2M roots, of which only ~59k are in the soma catalog) have no matching EM `DataItem` — `CellCellConnectivityLong` requires both endpoints to be registered DataItems, so the un-cataloged endpoints must be dropped or we must register additional "synapse-partner" DataItems for them. Leaning: drop the un-cataloged side and keep only edges where both endpoints are in `v1dd_1196_em`.
4. The label feather is indexed by synapse `id` but is shorter than the main synapse table (6.7M vs 8.2M) — unlabelled synapses should be treated as `tag=null`, not implicitly `non-spine`.
5. Connectome discriminator — per §4g, the folder name scopes the example; confirm `cellcellconnectivitylong_proofread_to_proofread/` is the right convention.

In [19]:
# TODO: load syn_df + syn_label_df; join labels on synapse id.
# TODO: groupby (pre, post) -> synapse_count, synapse_size_sum, spine_count.
# TODO: join pre/post pt_root_id -> EM DataItem id.
# TODO: build CellCellConnectivityLong rows (one per pair per measurement_type).
# TODO: write to cellcellconnectivitylong_proofread_to_proofread/.
pass

## 9. Functional cell-cell correlations — `CellCellConnectivityLong`

Two source tables, seven stimulus conditions each:
- `cell_cell_correlations_by_stimulus_coregistered.feather` — keyed by `pre_pt_root_id` / `post_pt_root_id` (which **are** the EM DataItem ids). Pairs **do** repeat (multiple ROIs per EM cell, ~4 %). Two options: (a) write rows directly as (EM, EM) pairs and let consumers see the duplicates, (b) average correlations within each (pre, post) pair, (c) explode back into functional DataItem ids via coreg and write at (func, func) level.
- `cell_cell_correlations_by_stimulus.feather` — keyed by `(volume, column, plane, roi)` × 2. Tuples are unique. Maps cleanly to functional DataItem ids.

Skeleton plan: one folder per (table, stimulus), e.g. `cellcellconnectivitylong_func_corr_drifting_gratings_full/`, `cellcellconnectivitylong_func_corr_coreg_drifting_gratings_full/`. 7 stimuli × 2 tables = 14 folders.

Verified earlier: in the coregistered table, 148728 rows reduce to 142410 unique (pre_root, post_root) pairs — i.e. ~4% of rows share a pair with another row. 12 self-pairs exist. Pre-set == post-set (551 cells, fully symmetric).

**Open questions:**
1. **Which key for the coregistered table?** With EM ids = pt_root_id, options (a)/(b)/(c) above are all on the table. (a) is the most direct; (b) loses ROI-level information; (c) requires picking which of the multiple coreg ROIs gets the correlation when collapsing pre side and same on post.
2. **Symmetry** — Pearson correlation is symmetric (corr(a,b) == corr(b,a)). The table appears to include both directions (8.8M rows ≈ N*(N-1) not N*(N-1)/2). Should we deduplicate, or keep as-is for ease of querying? `CellCellConnectivityLong` doesn't enforce direction.
3. **Self-pairs** — drop the 12 self-pair rows in the coregistered table?
4. **Measurement type / unit** — need a `SynapticMeasurementType` enum value for "Pearson correlation". If none exists, propose `pearson_correlation`?
5. **Scale** — 7 stimuli × 8.8M = 62M rows for the all-ROI table. Write all of it, or threshold (|r| > 0.1) and keep a sparse view? Storage cost vs query utility tradeoff.
6. Per §4g, do we need a per-stimulus folder, or can we use one folder with `measurement_type` as the discriminator? (Schema has only one `measurement_type` enum per row, so per-stimulus folders are the natural fit unless the enum has a `pearson_correlation_<stim>` variant — unlikely.)

In [20]:
corr_df    = pd.read_feather(DATA_ROOT / "cell_cell_correlations_by_stimulus.feather")
corr_co_df = pd.read_feather(DATA_ROOT / "cell_cell_correlations_by_stimulus_coregistered.feather")

print("== coregistered table ==")
print("rows                 :", len(corr_co_df))
print("unique pre roots     :", corr_co_df['pre_pt_root_id'].nunique())
print("unique post roots    :", corr_co_df['post_pt_root_id'].nunique())
print("unique (pre,post)    :", corr_co_df.drop_duplicates(['pre_pt_root_id','post_pt_root_id']).shape[0])
print("self pairs           :", (corr_co_df['pre_pt_root_id']==corr_co_df['post_pt_root_id']).sum())
print("pre set == post set  :", set(corr_co_df['pre_pt_root_id'].unique()) == set(corr_co_df['post_pt_root_id'].unique()))
print()
key = ['pre_roi','post_roi','pre_plane','post_plane','column','volume']
print("== all-ROI table ==")
print("rows                 :", len(corr_df))
print("unique tuples        :", corr_df.drop_duplicates(key).shape[0])
print("self pairs (same vol,col,pln,roi):",
      ((corr_df['pre_roi']==corr_df['post_roi']) & (corr_df['pre_plane']==corr_df['post_plane'])).sum())

== coregistered table ==
rows                 : 148728
unique pre roots     : 551
unique post roots    : 551
unique (pre,post)    : 142410
self pairs           : 12
pre set == post set  : True

== all-ROI table ==
rows                 : 8846260


unique tuples        : 8846260
self pairs (same vol,col,pln,roi): 0


In [21]:
# TODO: pivot each table -> long form (one row per pair per stimulus).
# TODO: resolve open questions above (key choice, dedup, threshold).
# TODO: write 14 folders cellcellconnectivitylong_func_corr_{coreg_,}<stimulus>/.
pass

## Summary (skeleton)

| Output path | Class | Rows | Status |
|---|---|---|---|
| `dataset/` | `DataSet` × 5 | 5 | done |
| `dataitem/` | `DataItem` | 163,063 EM + ~4.5k func (func TODO) | EM done, func skeleton |
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | 163,063 + cohort rows | EM done, func skeleton |
| `cellfeaturedefinition/`, `cellfeatureset/`, `cellfeaturematrix/`, `cellfeatures/v1dd_em_soma_geometry/` | EM soma geometry | 7 defs, 1 set, 1 matrix, 163,063 cell rows | done |
| `cluster/`, `clusterhierarchy/`, `algorithmrun/`, `clustermembership/` | V1DD taxonomy | 15, 1, 1, parent-propagated | skeleton |
| `cellfeatures/v1dd_func_qc/`, `cellfeatures/v1dd_func_imaging_position/` | Functional features | 1 + 4 defs, 2 sets, 2 matrices | skeleton |
| `mappingset/`, `celltocellmapping/` | EM↔func coregistration | 1 set, 571 rows | skeleton |
| `cellcellconnectivitylong_proofread_to_proofread/` | Synapse aggregation | ~N pairs × M measurement types | skeleton |
| `cellcellconnectivitylong_func_corr_<stim>/` × 7 | All-ROI correlations | ~8.8M per stim | skeleton |
| `cellcellconnectivitylong_func_corr_coreg_<stim>/` × 7 | Coreg correlations | ~149k per stim | skeleton |

**Intentionally not written**:
- `singlecellreconstruction/` — the V1DD release ships no per-cell reconstructions and the trform soma centroid is stored as features on `v1dd_em_soma_geometry` instead (see §3). Will be added if/when SWCs become available and a V1DD `reference_space` is pinned down.

**Cross-section open questions (need answers before we wire the writes):**
- Per-synapse schema vs aggregation-only — §8.
- Correlation key (EM cell vs functional ROI) and symmetry handling — §9.
